# Phase 2: baselines (zero-shot and few-shot), Qwen2.5-1.5B-Instruct

No training happens here. This notebook only runs inference and scores it, to get the **before** numbers for the fine-tune.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Upload `colab_bundle.zip` when asked (made by `python -m src.data.make_colab_bundle`).

Plan: run once with a small `N_DOCS_PER_LAYOUT` to **time it**, then raise it. Outputs are appended to a jsonl after every batch, so a disconnect loses at most one batch. Re-running the generation cell resumes.

In [ ]:
# 1. Setup
!pip -q install transformers accelerate pydantic
import torch, transformers, platform
print("torch", torch.__version__, "| transformers", transformers.__version__, "| python", platform.python_version())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch the runtime to T4")

In [ ]:
# 2. Upload and unpack the bundle
import zipfile, sys, os
from google.colab import files
if not os.path.exists("/content/work/data/pairs/val.jsonl"):
    up = files.upload()                                   # pick colab_bundle.zip
    zipfile.ZipFile(next(iter(up))).extractall("/content/work")
sys.path.insert(0, "/content/work")
os.chdir("/content/work")
from src.eval.prompts import build_messages, pick_shots
from src.eval.extraction_metrics import evaluate, format_table, write_csv, load_jsonl
print(os.listdir("data/pairs"))

In [ ]:
# 3. Config  (edit this cell only)
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SPLITS = ["val", "test"]            # C = val, D = test (held-out layouts)
KINDS = ["digital", "scan"]
MODES = ["zero", "few"]             # zero-shot, few-shot (2 examples from layouts A and B)
USE_TABLES = True                   # Phase 2 decision: append extracted tables. Set False for the text-only ablation.
N_DOCS_PER_LAYOUT = 20             # first run: 20 to measure speed. Then 100, then None (= all 300).
BATCH_SIZE = 8                      # raise until you hit out-of-memory, then back off
MAX_NEW_TOKENS = 768                # outputs that hit this limit are flagged 'truncated' and count as invalid
OUT_DIR = "/content/work/results"   # to survive disconnects: mount Drive and point this at a Drive folder
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 4. Optional: keep results on Google Drive so a disconnect cannot wipe them
# from google.colab import drive; drive.mount("/content/drive")
# OUT_DIR = "/content/drive/MyDrive/documentation-ai/results"; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# 5. Load model (fp16: the T4 has no fast bf16) and tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
tok = AutoTokenizer.from_pretrained(MODEL)
tok.padding_side = "left"           # decoder-only models must be LEFT padded for batched generate
model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float16, device_map="cuda")
model.eval()
print("pad:", tok.pad_token, "| eos:", tok.eos_token, "| params (M):", round(sum(p.numel() for p in model.parameters())/1e6))
print("model's default generation settings:", model.generation_config)   # check repetition_penalty etc.

In [ ]:
# 6. Load pairs and pick the fixed few-shot examples
def load(split): return load_jsonl(f"data/pairs/{split}.jsonl")
train = load("train")
shots_by_kind = {k: pick_shots(train, k, n=2) for k in KINDS}
for k, s in shots_by_kind.items():
    print(k, "shots:", [(p["id"], p["layout"]) for p in s])

def subset(split, kind):
    rows = [p for p in load(split) if p["kind"] == kind]
    if N_DOCS_PER_LAYOUT is None: return rows
    out, seen = [], {}
    for p in sorted(rows, key=lambda p: p["id"]):
        seen[p["layout"]] = seen.get(p["layout"], 0) + 1
        if seen[p["layout"]] <= N_DOCS_PER_LAYOUT: out.append(p)
    return out

# token budget check: prompt lengths in REAL tokens for the longest few-shot case
import statistics
def n_tokens(msgs):
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    return len(tok(text, add_special_tokens=False).input_ids)
probe = subset("val", "scan")[:10] + subset("test", "digital")[:10]
lens = [n_tokens(build_messages(p, shots=shots_by_kind[p["kind"]], tables=USE_TABLES)) for p in probe]
print(f"few-shot prompt tokens: mean {statistics.mean(lens):.0f}, max {max(lens)}  (model context is far larger, this is only a sanity check)")

In [ ]:
# 7. Generation. Resumable: ids already in the output file are skipped.
import json, time
EOS_IDS = {tok.eos_token_id, tok.pad_token_id}

@torch.no_grad()
def run(mode, split, kind):
    path = f"{OUT_DIR}/preds_{mode}_{split}_{kind}.jsonl"
    done = {json.loads(l)["id"] for l in open(path)} if os.path.exists(path) else set()
    rows = [p for p in subset(split, kind) if p["id"] not in done]
    shots = shots_by_kind[kind] if mode == "few" else ()
    prompts = {p["id"]: tok.apply_chat_template(build_messages(p, shots=shots, tables=USE_TABLES),
                                                tokenize=False, add_generation_prompt=True) for p in rows}
    rows.sort(key=lambda p: len(prompts[p["id"]]))         # similar lengths together = less padding waste
    t_all = time.time()
    with open(path, "a", encoding="utf-8") as f:
        for i in range(0, len(rows), BATCH_SIZE):
            batch = rows[i:i + BATCH_SIZE]
            enc = tok([prompts[p["id"]] for p in batch], return_tensors="pt", padding=True,
                      add_special_tokens=False).to("cuda")
            torch.cuda.synchronize(); t0 = time.time()
            out = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                 repetition_penalty=1.0,    # copying text exactly; a penalty would hurt
                                 pad_token_id=tok.pad_token_id)
            torch.cuda.synchronize(); dt = time.time() - t0
            gen = out[:, enc.input_ids.shape[1]:]
            for j, p in enumerate(batch):
                ids = gen[j]
                n_out = int((ids != tok.pad_token_id).sum())
                rec = {k: p[k] for k in ("id", "layout", "split", "kind", "target")}
                rec.update({"mode": mode, "raw": tok.decode(ids, skip_special_tokens=True),
                            "n_in_tokens": int(enc.attention_mask[j].sum()), "n_out_tokens": n_out,
                            "truncated": int(ids[-1]) not in EOS_IDS,
                            "batch_seconds": round(dt, 2), "batch_size": len(batch),
                            "tables_in_prompt": USE_TABLES, "model": MODEL})
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            f.flush()
            print(f"  {mode}/{split}/{kind}: {min(i+BATCH_SIZE, len(rows))}/{len(rows)} | last batch {dt:.1f}s")
    print(f"{mode}/{split}/{kind} finished in {time.time()-t_all:.0f}s | peak VRAM {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

for mode in MODES:
    for split in SPLITS:
        for kind in KINDS:
            run(mode, split, kind)

In [ ]:
# 8. Score everything in OUT_DIR
import glob
rows = [r for p in sorted(glob.glob(f"{OUT_DIR}/preds_*.jsonl")) for r in load_jsonl(p)]
results = evaluate(rows)
print(format_table(results))
for r in results:
    if r["fail_kinds"] or r["truncated"]:
        print(r["mode"], r["split"], r["layout"], r["kind"], "| invalid by kind:", r["fail_kinds"], "| truncated:", r["truncated"])
write_csv(results, f"{OUT_DIR}/baselines_{'tables' if USE_TABLES else 'textonly'}.csv")

In [ ]:
# 9. Look at failures with your own eyes (the numbers alone hide the cause)
import random
bad = [r for r in rows if r["mode"] == "zero" and r["split"] == "test"][:3]
for r in bad:
    print("=" * 80, "\n", r["id"], r["layout"], r["kind"], "| truncated:", r["truncated"])
    print("RAW :", r["raw"][:600]); print("GOLD:", r["target"][:600])

In [ ]:
# 10. Speed numbers worth recording (amortised over a batch: NOT single-document latency)
import statistics as st
for mode in MODES:
    rs = [r for r in rows if r["mode"] == mode]
    if rs:
        per_doc = [r["batch_seconds"] / r["batch_size"] for r in rs]
        tps = [r["n_out_tokens"] * r["batch_size"] / r["batch_seconds"] for r in rs]
        print(f"{mode}: {st.mean(per_doc):.2f} s/doc (batch {BATCH_SIZE}) | ~{st.mean(tps):.0f} generated tok/s per batch | mean prompt {st.mean(r['n_in_tokens'] for r in rs):.0f} tok")
print("peak VRAM (GB):", round(torch.cuda.max_memory_allocated()/1e9, 2))

**After the run:** download the `preds_*.jsonl` files and `baselines_*.csv` into `results/` in your repo (they are small), and fill the numbers into `docs/handoffs/phase2.md`. Do not edit the numbers by hand.